# Knowledge Graph Database Creation

This notebook creates a DuckDB database containing all the knowledge graph data from the FRED analysis.
The database will be used by the Dash application for interactive exploration.

In [ ]:
import pandas as pd
import duckdb
import os
from pathlib import Path
import numpy as np
import datetime

print(f"Starting database creation at: {datetime.datetime.now()}")

In [ ]:
# Set paths
base_path = Path('../2025_10_11_13_36_15_KnowledgeGraph_FRED_Analysis/outputs/csv')
db_path = Path('./knowledge_graph.db')

# Remove existing database if it exists
if db_path.exists():
    db_path.unlink()
    print(f"Removed existing database: {db_path}")

print(f"Data files location: {base_path}")
print(f"Database will be created at: {db_path}")

In [ ]:
# Check available CSV files
csv_files = list(base_path.glob('*.csv'))
print(f"Found {len(csv_files)} CSV files:")
for file in csv_files:
    print(f"  - {file.name} ({file.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Connect to DuckDB
conn = duckdb.connect(str(db_path))
print(f"Connected to DuckDB database: {db_path}")

# Create a metadata table
conn.execute("""
CREATE TABLE metadata (
    key VARCHAR,
    value VARCHAR,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

# Insert metadata
conn.execute("""
INSERT INTO metadata (key, value) VALUES 
    ('database_version', '1.0'),
    ('created_by', 'Knowledge Graph Analysis'),
    ('description', 'Economic indicators knowledge graph data from FRED analysis'),
    ('source_analysis', 'KnowledgeGraph_FRED_Analysis.ipynb')
""")

print("Created metadata table")

In [ ]:
# Load and process knowledge graph nodes
nodes_file = base_path / 'knowledge_graph_nodes.csv'

if nodes_file.exists():
    nodes_df = pd.read_csv(nodes_file)
    print(f"Loaded nodes data: {len(nodes_df)} rows, {len(nodes_df.columns)} columns")
    print(f"Columns: {list(nodes_df.columns)}")
    
    # Display first few rows
    print("\nFirst 5 rows:")
    print(nodes_df.head())
    
    # Create nodes table
    conn.execute("""
    CREATE TABLE nodes (
        id VARCHAR PRIMARY KEY,
        label VARCHAR,
        title VARCHAR,
        color VARCHAR,
        size DOUBLE,
        node_type VARCHAR,
        degree INTEGER,
        avg_strength DOUBLE,
        total_strength DOUBLE
    )
    """)
    
    # Insert nodes data
    conn.execute("INSERT INTO nodes SELECT * FROM nodes_df")
    
    node_count = conn.execute("SELECT COUNT(*) FROM nodes").fetchone()[0]
    print(f"\nInserted {node_count} nodes into database")
    
else:
    print(f"Nodes file not found: {nodes_file}")

In [ ]:
# Load and process knowledge graph edges
edges_file = base_path / 'knowledge_graph_edges.csv'

if edges_file.exists():
    edges_df = pd.read_csv(edges_file)
    print(f"Loaded edges data: {len(edges_df)} rows, {len(edges_df.columns)} columns")
    print(f"Columns: {list(edges_df.columns)}")
    
    # Display first few rows
    print("\nFirst 5 rows:")
    print(edges_df.head())
    
    # Create edges table
    conn.execute("""
    CREATE TABLE edges (
        from_node VARCHAR,
        to_node VARCHAR,
        weight DOUBLE,
        width DOUBLE,
        color VARCHAR,
        title VARCHAR,
        arrows VARCHAR
    )
    """)
    
    # Rename columns to match database schema
    edges_df = edges_df.rename(columns={
        'from': 'from_node',
        'to': 'to_node'
    })
    
    # Insert edges data
    conn.execute("INSERT INTO edges SELECT * FROM edges_df")
    
    edge_count = conn.execute("SELECT COUNT(*) FROM edges").fetchone()[0]
    print(f"\nInserted {edge_count} edges into database")
    
else:
    print(f"Edges file not found: {edges_file}")

In [ ]:
# Load and process correlations data
correlations_file = base_path / 'correlations_detailed.csv'

if correlations_file.exists():
    correlations_df = pd.read_csv(correlations_file)
    print(f"Loaded correlations data: {len(correlations_df)} rows, {len(correlations_df.columns)} columns")
    print(f"Columns: {list(correlations_df.columns)}")
    
    # Display first few rows
    print("\nFirst 5 rows:")
    print(correlations_df.head())
    
    # Create correlations table
    conn.execute("""
    CREATE TABLE correlations (
        x1 VARCHAR,
        x2 VARCHAR,
        best_lag INTEGER,
        best_corr DOUBLE,
        zero_lag_corr DOUBLE,
        n_obs INTEGER,
        correlation_type VARCHAR,
        lag_type VARCHAR,
        strength DOUBLE
    )
    """)
    
    # Insert correlations data
    conn.execute("INSERT INTO correlations SELECT * FROM correlations_df")
    
    corr_count = conn.execute("SELECT COUNT(*) FROM correlations").fetchone()[0]
    print(f"\nInserted {corr_count} correlations into database")
    
else:
    print(f"Correlations file not found: {correlations_file}")

In [ ]:
# Load and process leadership analysis data
leadership_file = base_path / 'leadership_analysis.csv'

if leadership_file.exists():
    leadership_df = pd.read_csv(leadership_file)
    print(f"Loaded leadership data: {len(leadership_df)} rows, {len(leadership_df.columns)} columns")
    print(f"Columns: {list(leadership_df.columns)}")
    
    # Display first few rows
    print("\nFirst 5 rows:")
    print(leadership_df.head())
    
    # Create leadership table
    conn.execute("""
    CREATE TABLE leadership (
        x1 VARCHAR PRIMARY KEY,
        total_connections INTEGER,
        avg_lag DOUBLE,
        median_lag DOUBLE,
        leading_connections INTEGER,
        coincident_connections INTEGER,
        lagging_connections INTEGER,
        avg_correlation_strength DOUBLE,
        leadership_score DOUBLE,
        indicator_role VARCHAR,
        indicator_category VARCHAR
    )
    """)
    
    # Insert leadership data
    conn.execute("INSERT INTO leadership SELECT * FROM leadership_df")
    
    leadership_count = conn.execute("SELECT COUNT(*) FROM leadership").fetchone()[0]
    print(f"\nInserted {leadership_count} leadership records into database")
    
else:
    print(f"Leadership file not found: {leadership_file}")

In [ ]:
# Load and process stock-economic relationships
stock_relationships_file = base_path / 'stock_economic_relationships.csv'

if stock_relationships_file.exists():
    stock_df = pd.read_csv(stock_relationships_file)
    print(f"Loaded stock relationships data: {len(stock_df)} rows, {len(stock_df.columns)} columns")
    print(f"Columns: {list(stock_df.columns)}")
    
    # Display first few rows
    print("\nFirst 5 rows:")
    print(stock_df.head())
    
    # Create stock_relationships table
    conn.execute("""
    CREATE TABLE stock_relationships (
        x1 VARCHAR,
        x2 VARCHAR,
        best_lag INTEGER,
        best_corr DOUBLE,
        zero_lag_corr DOUBLE,
        n_obs INTEGER,
        correlation_type VARCHAR,
        lag_type VARCHAR,
        strength DOUBLE,
        stock_symbol VARCHAR,
        economic_indicator VARCHAR,
        stock_leads BOOLEAN,
        stock_type VARCHAR,
        econ_category VARCHAR
    )
    """)
    
    # Insert stock relationships data
    conn.execute("INSERT INTO stock_relationships SELECT * FROM stock_df")
    
    stock_count = conn.execute("SELECT COUNT(*) FROM stock_relationships").fetchone()[0]
    print(f"\nInserted {stock_count} stock relationship records into database")
    
else:
    print(f"Stock relationships file not found: {stock_relationships_file}")

In [ ]:
# Create useful views for the Dash application

# View for strong correlations with node information
conn.execute("""
CREATE VIEW strong_correlations_with_nodes AS
SELECT 
    c.*,
    n1.node_type as x1_type,
    n1.degree as x1_degree,
    n2.node_type as x2_type,
    n2.degree as x2_degree
FROM correlations c
LEFT JOIN nodes n1 ON c.x1 = n1.id
LEFT JOIN nodes n2 ON c.x2 = n2.id
WHERE ABS(c.best_corr) >= 0.7
""")

# View for network statistics by node type
conn.execute("""
CREATE VIEW node_type_stats AS
SELECT 
    node_type,
    COUNT(*) as node_count,
    AVG(degree) as avg_degree,
    MAX(degree) as max_degree,
    AVG(avg_strength) as avg_correlation_strength
FROM nodes
WHERE node_type IS NOT NULL
GROUP BY node_type
ORDER BY avg_degree DESC
""")

# View for top correlations by category
conn.execute("""
CREATE VIEW top_correlations_by_type AS
SELECT 
    x1_type,
    x2_type,
    COUNT(*) as relationship_count,
    AVG(ABS(best_corr)) as avg_correlation,
    AVG(best_lag) as avg_lag
FROM strong_correlations_with_nodes
WHERE x1_type IS NOT NULL AND x2_type IS NOT NULL
GROUP BY x1_type, x2_type
HAVING COUNT(*) >= 5
ORDER BY avg_correlation DESC
""")

print("Created database views for the Dash application")

In [ ]:
# Database summary and validation
print("\n" + "="*60)
print("DATABASE SUMMARY")
print("="*60)

# Show all tables
tables = conn.execute("SHOW TABLES").fetchall()
print(f"\nTables created: {len(tables)}")
for table in tables:
    table_name = table[0]
    count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"  - {table_name}: {count:,} records")

# Show views created
views = conn.execute("SELECT name FROM sqlite_master WHERE type='view'").fetchall()
print(f"\nViews created: {len(views)}")
for view in views:
    view_name = view[0]
    count = conn.execute(f"SELECT COUNT(*) FROM {view_name}").fetchone()[0]
    print(f"  - {view_name}: {count:,} records")

print(f"\nDatabase file size: {db_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"Database location: {db_path.absolute()}")

In [ ]:
# Test some sample queries that the Dash app will use
print("\n" + "="*60)
print("SAMPLE QUERIES FOR DASH APPLICATION")
print("="*60)

# Query 1: Node type distribution
print("\n1. Node type distribution:")
result = conn.execute("SELECT * FROM node_type_stats LIMIT 10").df()
print(result.to_string())

# Query 2: Top correlations
print("\n2. Top 10 correlations:")
result = conn.execute("""
SELECT x1, x2, best_corr, lag_type 
FROM correlations 
ORDER BY ABS(best_corr) DESC 
LIMIT 10
""").df()
print(result.to_string())

# Query 3: Leadership rankings
print("\n3. Top 10 economic leaders:")
result = conn.execute("""
SELECT x1, indicator_category, leadership_score, indicator_role
FROM leadership 
ORDER BY leadership_score DESC 
LIMIT 10
""").df()
print(result.to_string())

In [ ]:
# Close database connection
conn.close()
print(f"\nDatabase creation completed successfully!")
print(f"Database saved as: {db_path.absolute()}")
print(f"Completed at: {datetime.datetime.now()}")

# Final validation - reconnect and verify
test_conn = duckdb.connect(str(db_path))
tables_count = len(test_conn.execute("SHOW TABLES").fetchall())
test_conn.close()

print(f"\nValidation: Database contains {tables_count} tables and is ready for use!")